In [ ]:
import os
from openai import OpenAI
from rich.console import Console
import json

In [ ]:
client = OpenAI(base_url="https://api.groq.com/openai/v1",api_key=os.getenv("GROQ_API_KEY"),)

In [ ]:
#create my own checklist agent from scratch
def show(text):
    try:
        Console().print(text)
    except:
        print(text)

In [ ]:
checklist =[]
completed = []

In [ ]:
def get_checklist_report():
    result = ""
    for idx, ele in enumerate(checklist):
        if completed[idx]:
            result += f"Completed {idx+1} : [green][strike]{ele}[/strike][/green]\n"
        else:
            result += f"Pending {idx+1} : {ele}\n"
    show(result)
    return result


In [ ]:
def create_checklist(descriptions):
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()

In [ ]:
def mark_complete(index, completion_notes):
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "Index Out Of Bounds"
    show(completion_notes)
    return get_checklist_report()


In [ ]:
create_checklist(["eat","sleep","repeat"])
print(checklist)


In [ ]:
mark_complete(2,"complete task 2")

In [ ]:
#decalre the tool to pass to client
create_checklist_json = {
    "name": "create_checklist",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions of checklist items'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [ ]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [ ]:
tools = [{"type": "function", "function": create_checklist_json},
        {"type": "function", "function": mark_complete_json}]

In [ ]:
def handle_tool_calls(tool_calls):
    results=[]
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [ ]:
def loop(messages):
    response = client.chat.completions.create(model = "openai/gpt-oss-120b",messages = messages, tools = tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        result = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(result)
        response = client.chat.completions.create(model="openai/gpt-oss-120b", messages=messages, tools=tools)
    show(response.choices[0].message.content)




In [ ]:
system_message = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A solid wooden cube of side 6 cm is painted blue on two adjacent faces, red on the faces opposite to the blue ones, and green on the remaining two faces.
The cube is then cut into 216 identical smaller cubes of side 1 cm each. How many of these smaller cubes have at least one blue face AND at least one green face, while having NO red face?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [ ]:
checklist=[]
completed=[]
loop(messages)